In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ML 'random_forest' for BoW:

In [42]:
print("="*50)
print("LOADING DATA")
print("="*50)

df_truth = pd.read_csv('GROUBD_TRUTH_WITH_FINAL_LABEL.csv')
print(f"Ground Truth: {df_truth.shape} rows")

df_scheme_a = pd.read_csv('bow_scheme_a.csv', index_col=0)
df_scheme_b = pd.read_csv('bow_scheme_b.csv', index_col=0)
df_scheme_c = pd.read_csv('bow_scheme_c.csv', index_col=0)

print(f"Scheme A: {df_scheme_a.shape}")
print(f"Scheme B: {df_scheme_b.shape}")
print(f"Scheme C: {df_scheme_c.shape}")

print("\n" + "="*50)
print("ALIGNING DATA")
print("="*50)

min_rows = min(len(df_scheme_a), len(df_scheme_b), len(df_scheme_c), len(df_truth))
print(f"Using {min_rows} rows")

df_scheme_a = df_scheme_a.iloc[:min_rows].copy()
df_scheme_b = df_scheme_b.iloc[:min_rows].copy()
df_scheme_c = df_scheme_c.iloc[:min_rows].copy()
df_truth_aligned = df_truth.iloc[:min_rows].copy()

df_scheme_a['final_label'] = df_truth_aligned['final_label'].values
df_scheme_b['final_label'] = df_truth_aligned['final_label'].values
df_scheme_c['final_label'] = df_truth_aligned['final_label'].values

print("\n" + "="*50)
print("LABEL ENCODING")
print("="*50)

encoder = LabelEncoder()

df_scheme_a['label_encoded'] = encoder.fit_transform(df_scheme_a['final_label'])
df_scheme_b['label_encoded'] = encoder.transform(df_scheme_b['final_label'])
df_scheme_c['label_encoded'] = encoder.transform(df_scheme_c['final_label'])

print(f"Classes: {encoder.classes_}")

print("\n" + "="*50)
print("RANDOM FOREST WITH BEST PARAMETERS")
print("="*50)

def random_forest_best(df, scheme_name):
    print(f"\n--- {scheme_name} ---")
    
    X = df.drop(['final_label', 'label_encoded'], axis=1).values
    y = df['label_encoded'].values
    
    original_features = X.shape[1]
    print(f"Original features: {original_features}")
    
    selector = VarianceThreshold(threshold=0)
    X = selector.fit_transform(X)
    print(f"Features after removing zeros: {X.shape[1]}")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    
    cv_scores = cross_val_score(rf, X_train, y_train, cv=5)
    print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"Test Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))
    
    return acc, rf

acc_a, rf_a = random_forest_best(df_scheme_a, "Scheme A")
acc_b, rf_b = random_forest_best(df_scheme_b, "Scheme B")
acc_c, rf_c = random_forest_best(df_scheme_c, "Scheme C")

print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)

print(f"Scheme A Accuracy: {acc_a:.4f}")
print(f"Scheme B Accuracy: {acc_b:.4f}")
print(f"Scheme C Accuracy: {acc_c:.4f}")

best_acc = max(acc_a, acc_b, acc_c)
best_scheme = ['Scheme A', 'Scheme B', 'Scheme C'][[acc_a, acc_b, acc_c].index(best_acc)]

print(f"\nBest scheme: {best_scheme} with {best_acc:.4f} accuracy")
print(f"Best accuracy: {best_acc*100:.2f}%")

LOADING DATA
Ground Truth: (200, 21) rows
Scheme A: (198, 460)
Scheme B: (197, 447)
Scheme C: (194, 340)

ALIGNING DATA
Using 194 rows

LABEL ENCODING
Classes: ['negative' 'neutral' 'positive']

RANDOM FOREST WITH BEST PARAMETERS

--- Scheme A ---
Original features: 460
Features after removing zeros: 459
Cross-validation accuracy: 0.5032 (+/- 0.0158)
Test Accuracy: 0.4872

Classification Report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        10
     neutral       0.00      0.00      0.00         9
    positive       0.50      0.95      0.66        20

    accuracy                           0.49        39
   macro avg       0.17      0.32      0.22        39
weighted avg       0.26      0.49      0.34        39


--- Scheme B ---
Original features: 447
Features after removing zeros: 447
Cross-validation accuracy: 0.5032 (+/- 0.0158)
Test Accuracy: 0.5128

Classification Report:
              precision    recall  f1-score   suppo

# ML 'random_forest' for GloVe:

In [44]:
df1 = pd.read_csv('glove_schema1.csv')
df2 = pd.read_csv('glove_schema2.csv')
df3 = pd.read_csv('glove_schema3.csv')
df_target = pd.read_csv('GROUBD_TRUTH_WITH_FINAL_LABEL.csv')

def extract_embeddings(df):
    embeddings = []
    for emb in df['embedding']:
        if isinstance(emb, str):
            vec = [float(x) for x in emb.split(',')]
            embeddings.append(vec)
        else:
            embeddings.append(emb)
    return np.array(embeddings)

emb1 = extract_embeddings(df1)
emb2 = extract_embeddings(df2)
emb3 = extract_embeddings(df3)

min_rows = min(emb1.shape[0], emb2.shape[0], emb3.shape[0], len(df_target))

emb1 = emb1[:min_rows]
emb2 = emb2[:min_rows]
emb3 = emb3[:min_rows]
df_target = df_target.iloc[:min_rows]

y_raw = df_target['final_label'].values
le = LabelEncoder()
y = le.fit_transform(y_raw)

results = {}
reports = {}

print("="*60)
print("CLASSIFICATION REPORT FOR EACH FILE")
print("="*60)

files = {
    'File 1 (glove_schema1)': emb1,
    'File 2 (glove_schema2)': emb2,
    'File 3 (glove_schema3)': emb3
}

for name, X in files.items():
    X_clean = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y, test_size=0.2, random_state=42, stratify=y
    )
    
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    results[name] = acc
    reports[name] = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\n" + "="*60)
print("BEST MODEL BASED ON ACCURACY")
print("="*60)

best_name = max(results, key=results.get)
best_acc = results[best_name]

print(f"\nBest Model: {best_name}")
print(f"Best Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"\nBest Model Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\n" + "="*60)
print("ALL RESULTS SUMMARY")
print("="*60)
for name, acc in results.items():
    print(f"{name:<25} : {acc:.4f} ({acc*100:.2f}%)")

CLASSIFICATION REPORT FOR EACH FILE

File 1 (glove_schema1)
Accuracy: 0.4872 (48.72%)

Classification Report:
              precision    recall  f1-score   support

    negative       0.33      0.10      0.15        10
     neutral       0.00      0.00      0.00         9
    positive       0.51      0.90      0.65        20

    accuracy                           0.49        39
   macro avg       0.28      0.33      0.27        39
weighted avg       0.35      0.49      0.38        39


File 2 (glove_schema2)
Accuracy: 0.4615 (46.15%)

Classification Report:
              precision    recall  f1-score   support

    negative       0.25      0.10      0.14        10
     neutral       0.00      0.00      0.00         9
    positive       0.52      0.85      0.64        20

    accuracy                           0.46        39
   macro avg       0.26      0.32      0.26        39
weighted avg       0.33      0.46      0.37        39


File 3 (glove_schema3)
Accuracy: 0.4359 (43.59%)

Cla